In [1]:
import glob, os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

path_to_validation = '/home/gterren/dynamic_update/validation/'
path_to_test       = '/home/gterren/dynamic_update/test/'
path_to_params     = '/home/gterren/dynamic_update/params/'

In [8]:
resource = 'solar'
method   = 'fusion'

hyper_ = pd.read_csv(path_to_params + f"{resource}/{resource}-{method}-hyper-sa.csv", low_memory=False) 
print(hyper_)

csv_ = glob.glob(path_to_validation + f"{resource}/{resource}-validation_ffc-sa.csv")
val_ = pd.read_csv(csv_[0], low_memory=False)
val_ = val_.loc[val_['method'] == method].reset_index(drop = True)

times_  = val_['time'].unique()
params_ = val_['parameter'].unique()

params_ = ['p_fusion']
times_  = [120, 144, 168]

for param in [params_[0]]:  
    for time in times_:
        print(time, param)
        idx_ = (val_['parameter'] == param) & (val_['time'] == time)
        val_p_ = val_.loc[idx_].reset_index(drop = True)
        val_p_ = val_p_[['value', 
                         'time', 
                         'threshold',
                         'WIS',
                         'KS', 
                         'MBE',
                         'RMSE']]
        
        print(val_p_)

         parameter       120       132       144       168
0    forget_rate_f    0.0625    0.0625    0.0625    0.0625
1    forget_rate_e    1.0000    0.7500    0.5000    0.2500
2      lookup_rate   64.0000   48.0000   32.0000   16.0000
3   length_scale_f    0.0100    0.0175    0.0250    0.0750
4   length_scale_e    0.5000    0.5000    0.5000    0.5000
5       trust_rate    0.5000    0.4500    0.4000    0.3000
6               nu    6.0000    7.0000    8.0000   12.0000
7               xi    0.5000    0.5500    0.6000    0.8000
8            gamma   20.0000   20.0000   20.0000   20.0000
9            kappa  175.0000  175.0000  175.0000  175.0000
10        p_fusion    0.7500    0.7500    0.7500    0.7500
120 p_fusion
    value  time  threshold       WIS        KS       MBE      RMSE
0    0.10   120    0.10854  0.039823  0.400716  0.005592  0.089397
1    0.20   120    0.10854  0.039856  0.389440  0.005449  0.089470
2    0.30   120    0.10854  0.039788  0.380466  0.005662  0.089518
3    0.40  

# Test Results

In [ ]:
# methods_ = ["fusion", "dayahead", "observation"]

# dfs_ = []
# for resource in ['solar']:
#     for method in methods_:
#         df_ = pd.read_csv(path_to_test + f"{resource}-{method}-iter_1-test_ffc.csv", low_memory=False)
#         df_ = df_.groupby(['time']).agg({'WIS_f': 'median'}).reset_index(drop = False)

#         df_['resource'] = resource
#         df_['method']   = method

#         dfs_.append(df_)

# dfs_ = pd.concat(dfs_, ignore_index=True)
# print(dfs_)
# # 0.03678
# pivot_ = dfs_.pivot_table(index=["method"],              # rows
#                           columns=["resource", "time"],  # columns
#                           values="WIS_f")                # cell values

# # optional: bring the index back as columns
# #pivot_ = pivot_.reset_index()
# # pivot_ = pivot_[["observation", "dayahead", "fusion"]]
# # print(pivot_)

# pivot_ = pivot_.sort_values(["method"], key = lambda s: pd.Categorical(s, categories=methods_, ordered=True))
# print(pivot_)

   time     WIS_f resource       method
0   120  0.035963    solar       fusion
1   144  0.029698    solar       fusion
2   168  0.022275    solar       fusion
3   120  0.040965    solar     dayahead
4   144  0.033251    solar     dayahead
5   168  0.025421    solar     dayahead
6   120  0.036226    solar  observation
7   144  0.030095    solar  observation
8   168  0.023055    solar  observation
resource        solar                    
time              120       144       168
method                                   
fusion       0.035963  0.029698  0.022275
dayahead     0.040965  0.033251  0.025421
observation  0.036226  0.030095  0.023055


# Functional Envelop Results

In [ ]:
from math import dist

resource  = 'solar'
method = 'fusion'
score = 'FIS'

df_ = pd.read_csv(path_to_validation + f'{resource}/{resource}-validation_envelop-0.csv', low_memory=False)

df_init_ = pd.read_csv(path_to_params + f"{resource}/{resource}-{method}-envelop.csv", low_memory=False)

for dist in df_['distance'].unique():
    for alpha in df_['alpha'].unique():
        for time in df_['time'].unique():
            idx_ = (df_['time'] == time) & (df_['alpha'] == alpha) & (df_['distance'] == dist)

            df_time = df_[idx_].reset_index(drop = True)
            df_time = df_time.groupby(['time', 
                                       'alpha', 
                                       'distance', 
                                       'fraction']).agg({score: 'median'}).reset_index(drop = False)

            if score == 'FCS':
                df_time[score] = (df_time[score] - (1 - alpha))**2

            idx_ = ((df_init_['alpha'] == alpha) & (df_init_['time'] == time) & (df_init_['distance'] == dist))

            df_init_.loc[idx_, 'fraction'] = df_time.loc[df_time[score].argmin(), 'fraction']
            #print(dist, alpha, time, df_time.loc[df_time[score].argmin(), 'fraction'])

print(df_init_)

df_init_.to_csv(path_to_params + f"{resource}-{method}-envelop-{score}.csv", index = False)

    time  alpha  fraction distance
0    120    0.1      0.70     fknn
1    120    0.2      0.40     fknn
2    120    0.3      0.25     fknn
3    120    0.4      0.20     fknn
4    120    0.1      0.80       l2
5    120    0.2      0.50       l2
6    120    0.3      0.30       l2
7    120    0.4      0.20       l2
8    120    0.1      0.80      sup
9    120    0.2      0.50      sup
10   120    0.3      0.30      sup
11   120    0.4      0.20      sup
12   144    0.1      0.60     fknn
13   144    0.2      0.30     fknn
14   144    0.3      0.20     fknn
15   144    0.4      0.15     fknn
16   144    0.1      0.70       l2
17   144    0.2      0.40       l2
18   144    0.3      0.25       l2
19   144    0.4      0.20       l2
20   144    0.1      0.60      sup
21   144    0.2      0.40      sup
22   144    0.3      0.25      sup
23   144    0.4      0.15      sup
24   168    0.1      0.50     fknn
25   168    0.2      0.30     fknn
26   168    0.3      0.20     fknn
27   168    0.4     

In [ ]:
# method = 'fusion'
# score  = 'FCS'

# dfs_ = []
# for resource in ['wind']:
#     for dist in ["fknn", "l2", "sup"]:
#         df_ = pd.read_csv(path_to_test + f"{resource}-{method}-{dist}-{score}_test_envelop_iter_1.csv", 
#                           low_memory = False)

#         df_ = df_.groupby(['alpha', 
#                            'time']).agg({score: 'median', 
#                                          'M' + score[1:]: 'median'}).reset_index(drop = False)

#         df_['resource'] = resource
#         df_['method']   = method
#         df_['distance'] = dist

#         dfs_.append(df_)

# dfs_ = pd.concat(dfs_, ignore_index=True)

# pivot_ = dfs_.pivot_table(index   = ['distance'],       # rows
#                           columns = ['time', 'alpha'],  # columns
#                           values  = score)              # cell values
# print(pivot_)

# pivot_ = dfs_.pivot_table(index   = ['distance'],       # rows
#                           columns = ['time', 'alpha'],  # columns
#                           values  =  'M' + score[1:])              # cell values
# print(pivot_)

time             72                                          144             \
alpha            0.1        0.2        0.3        0.4        0.1        0.2   
distance                                                                      
fknn      111.987244  96.491746  85.249457  77.092200  75.020353  64.766013   
l2        112.077484  94.353226  82.854213  75.139104  75.382931  62.366965   
sup       112.563568  93.925933  82.725449  74.899342  74.926550  62.705141   

time                                  216                                   
alpha           0.3        0.4        0.1        0.2        0.3        0.4  
distance                                                                    
fknn      57.196414  51.241733  38.653400  32.656770  28.612153  25.604044  
l2        54.975814  49.719701  38.684899  32.243191  28.158605  25.248820  
sup       55.548273  49.702395  38.630264  32.554533  28.345770  25.297114  
time             72                                          14

# Functional Depth Results

In [5]:
from math import dist

resource  = 'solar'
method = 'fusion'
score = 'FIS'

df_ = pd.read_csv(path_to_validation + f'{resource}/{resource}-validation_depth-0.csv', low_memory=False)

df_init_ = pd.read_csv(path_to_params + f"{resource}/{resource}-{method}-depth.csv", low_memory=False)

for dist in df_['distance'].unique():
    for alpha in df_['alpha'].unique():
        for time in df_['time'].unique():
            idx_ = (df_['time'] == time) & (df_['alpha'] == alpha) & (df_['distance'] == dist)

            df_time = df_[idx_].reset_index(drop = True)
            df_time = df_time.groupby(['time', 
                                       'alpha', 
                                       'distance', 
                                       'fraction']).agg({score: 'median'}).reset_index(drop = False)

            if score == 'FCS':
                df_time[score] = (df_time[score] - (1 - alpha))**2

            idx_ = ((df_init_['alpha'] == alpha) & (df_init_['time'] == time) & (df_init_['distance'] == dist))

            df_init_.loc[idx_, 'fraction'] = df_time.loc[df_time[score].argmin(), 'fraction']
            print(dist, alpha, time, df_time.loc[df_time[score].argmin(), 'fraction'])

print(df_init_)

df_init_.to_csv(path_to_params + f"{resource}-{method}-depth-{score}.csv", index = False)

MBD 0.1 168 0.5
MBD 0.1 144 0.5
MBD 0.1 120 0.6
MBD 0.2 168 0.3
MBD 0.2 144 0.3
MBD 0.2 120 0.4
MBD 0.3 168 0.15
MBD 0.3 144 0.15
MBD 0.3 120 0.15
MBD 0.4 168 0.1
MBD 0.4 144 0.1
MBD 0.4 120 0.15
    time  alpha  fraction distance
0    120    0.1      0.60      MBD
1    144    0.1      0.50      MBD
2    168    0.1      0.50      MBD
3    120    0.2      0.40      MBD
4    144    0.2      0.30      MBD
5    168    0.2      0.30      MBD
6    120    0.3      0.15      MBD
7    144    0.3      0.15      MBD
8    168    0.3      0.15      MBD
9    120    0.4      0.15      MBD
10   144    0.4      0.10      MBD
11   168    0.4      0.10      MBD


In [ ]:

df_ = pd.read_csv(path_to_validation + r'wind_fusion_dispersion_ts.csv', low_memory=False)
#print(df_)

df_filtered_ = df_.loc[(df_['error'] > 0.) & (df_['error'] < .125)].reset_index(drop = True)
df_filtered_['dummy'] = 1

df_filtered_ = df_filtered_.loc[df_filtered_['interval'] == 144]

df_filtered_ = df_filtered_.groupby(['asset', 'day',]).agg({'std': 'mean', 
                                                            'mean': 'mean',
                                                            'error': 'mean',
                                                            'dummy': 'sum'}).reset_index(drop = False)

#df_filtered_ = df_filtered_.loc[df_filtered_['dummy'] == 3].reset_index(drop = True)

idx_ = (df_filtered_['mean'] > 0.6) & (df_filtered_['mean'] < 0.8)

df_filtered_ = df_filtered_.loc[idx_].reset_index(drop = True).sort_values('std', ascending = False)

print(df_filtered_.head(30))

     asset  day       std      mean     error  dummy
149     34   54  0.230047  0.725602  0.121388      1
48      25  316  0.229516  0.665538  0.093726      1
39      25  122  0.215829  0.645466  0.087989      1
178     38   20  0.214496  0.736086  0.117132      1
65      27  293  0.213951  0.652661  0.089004      1
72      28   69  0.212569  0.703793  0.117249      1
160     36   54  0.211422  0.694022  0.090698      1
147     33  285  0.208866  0.698756  0.085554      1
81      29   69  0.208819  0.786321  0.068078      1
140     33  122  0.207436  0.693730  0.111810      1
156     34  262  0.207344  0.618695  0.119742      1
37      25  119  0.199534  0.660502  0.079610      1
19      22   45  0.198674  0.676025  0.109753      1
139     33  119  0.197882  0.668325  0.077584      1
9       20  347  0.197511  0.738598  0.113698      1
50      26   69  0.195914  0.704686  0.088393      1
66      27  306  0.195503  0.635529  0.114656      1
78      28  353  0.195189  0.757241  0.100887 

In [ ]:

df_ = pd.read_csv(path_to_validation + r'wind_fusion_dispersion_ts.csv', low_memory=False)
print(df_['error'].min(), df_['error'].max())

# Pivot std
std_pivot = df_.pivot_table(
    index=['asset', 'day'],
    columns='interval',
    values='std'
)

# Pivot error
error_pivot = df_.pivot_table(
    index=['asset', 'day'],
    columns='interval',
    values='mean'
)

mask = (
    (std_pivot[72] > std_pivot[144]) &
    (std_pivot[144] > std_pivot[216])
)

filtered = std_pivot[mask]

# Compute max error across intervals
filtered['mean'] = error_pivot.loc[filtered.index][[72, 144, 216]].mean(axis=1)

result = filtered.reset_index().sort_values('mean')
print(result.loc[result['mean'] > 0.6].head(20))

0.0 0.642702472683188
interval  asset  day        72       144       216      mean
756          39    4  0.159368  0.124906  0.090452  0.100882
173          24   17  0.147505  0.147039  0.084130  0.101108
651          35   51  0.145020  0.129104  0.115688  0.101574
763          39   15  0.171555  0.134177  0.075524  0.103542
653          35   63  0.177398  0.120636  0.112230  0.105403
530          33    4  0.120485  0.120249  0.119195  0.108720
177          24   22  0.172518  0.142594  0.077484  0.109191
658          35   79  0.166874  0.163693  0.138205  0.109363
188          24   64  0.168521  0.140445  0.119504  0.111699
761          39   13  0.152618  0.144726  0.086644  0.111940
176          24   21  0.166471  0.158104  0.076132  0.113740
185          24   41  0.179735  0.145014  0.099895  0.114707
538          33   18  0.151981  0.142509  0.109890  0.120623
182          24   31  0.169984  0.150239  0.137559  0.120705
127          23    1  0.175684  0.141666  0.068197  0.121462
18

/tmp/ipykernel_4156045/546663543.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered['mean'] = error_pivot.loc[filtered.index][[72, 144, 216]].mean(axis=1)


In [1]:

df_ = pd.read_csv(path_to_validation + r'wind_fusion_dispersion_ts.csv', low_memory=False)
print(df_['error'].min(), df_['error'].max())

# Pivot std
std_pivot = df_.pivot_table(
    index=['asset', 'day'],
    columns='interval',
    values='std'
)

# Pivot error
error_pivot = df_.pivot_table(
    index=['asset', 'day'],
    columns='interval',
    values='error'
)

mask = (
    (std_pivot[120] > std_pivot[144]) &
    (std_pivot[144] > std_pivot[168])
)

filtered = std_pivot[mask]

# Compute max error across intervals
filtered['error'] = error_pivot.loc[filtered.index][[120, 144, 168]].mean(axis=1)

result = filtered.reset_index().sort_values('error')
print(result.loc[result['error'] > 0.1].head(20))

NameError: name 'pd' is not defined

In [ ]:
df_ = pd.read_csv(path_to_validation + r'solar_fusion_dispersion_ts.csv', low_memory=False)
for interval in df_['interval'].unique():
    df_interval = df_.loc[df_['interval'] == interval].reset_index(drop = True)
    
    idx_1 = df_interval['n_spatial'] == df_interval['n_spatial'].min()
    idx_2 = (df_interval['n_spatial'] > df_interval['n_spatial'].min()) & (df_interval['n_spatial'] < df_interval['n_spatial'].max())
    idx_3 = df_interval['n_spatial'] == df_interval['n_spatial'].max()

    print(interval, idx_1.sum(), idx_2.sum(), idx_3.sum())


# wind test:
# interval n<min min<n<max n>max
#   72      256     183     6821
#   144     564     398     6298
#   216     3777    439     3044
# solar test:
# interval n<min min<n<max n>max
#   120 586 23 6651
#   144 431 34 6795
#   168 789 70 6401

120 586 23 6651
144 431 34 6795
168 789 70 6401
